# Temporal KG Evaluation V4 — Final Methodology Candidate

**Purpose:** Auditable I01 instance-aligned temporal event evaluation. This is a candidate for freezing after Colab execution and review, not a guarantee of deployment quality.

**Scope:** Retain all 60 original Jira tickets for traceability; evaluate only tickets with documented I01-window activity and events within those windows. **Both** reference and graph event timestamps use the same boundaries.

**Predeclared vocabulary:** Only justified mappings are scored. Unmapped event types are reported as coverage limitations. Strict matching requires issue key + mapped event type + exact UTC timestamp, one-to-one by multiplicity.

**Interpretation:** Event F1 is conditional on supported categories and in-scope events. The timestamp-derived ordering check is a *consistency diagnostic*, not an independent accuracy test of temporal edges. Availability metadata is not proof of leakage-free downstream risk prediction.

**Integrity:** Source files and graph exports are read-only. Results are written into a new `Results/V4_Final_Methodology` folder; V3 outputs are preserved.


## 0. Input configuration

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, zipfile, collections, hashlib
import pandas as pd, numpy as np
from IPython.display import display

ROOT=Path('/content/drive/MyDrive')
GC1=ROOT/'GC1'
BASE=ROOT/'01_Knowledge_Construction'/'Temporal_KG_Evaluation'
GT=BASE/'Ground_Truth'; OUT=BASE/'Results'/'V4_Final_Methodology'
OUT.mkdir(parents=True,exist_ok=True)
JIRA=GC1/'GC1_Jira_Dataset'
PRD=GC1/'GC1_Generated_PRD_Meetings'
GRAPH=GC1/'entity_resolution_output'/'temporal_kg_v2_output'
PROJECTS=['HBASE','JBIDE','MC','MNG','SAK','SERVER']
for title,p in [('Ground truth events',GT/'temporal_events_reference.csv'),('Ground truth relations',GT/'temporal_relations_reference.csv'),('Jira input',JIRA),('Manifests',PRD),('Graph export',GRAPH)]:
    print(title, 'FOUND' if p.exists() else 'MISSING',p)
    if not p.exists():raise FileNotFoundError(str(p))
print('Results:',OUT)

Mounted at /content/drive
Ground truth events FOUND /content/drive/MyDrive/01_Knowledge_Construction/Temporal_KG_Evaluation/Ground_Truth/temporal_events_reference.csv
Ground truth relations FOUND /content/drive/MyDrive/01_Knowledge_Construction/Temporal_KG_Evaluation/Ground_Truth/temporal_relations_reference.csv
Jira input FOUND /content/drive/MyDrive/GC1/GC1_Jira_Dataset
Manifests FOUND /content/drive/MyDrive/GC1/GC1_Generated_PRD_Meetings
Graph export FOUND /content/drive/MyDrive/GC1/entity_resolution_output/temporal_kg_v2_output
Results: /content/drive/MyDrive/01_Knowledge_Construction/Temporal_KG_Evaluation/Results/V4_Final_Methodology


## 1. Load frozen references and check integrity

In [2]:
ref=pd.read_csv(GT/'temporal_events_reference.csv',dtype=str).fillna('')
rels=pd.read_csv(GT/'temporal_relations_reference.csv',dtype=str).fillna('')
assert {'event_id','issue_key','project_key','event_type','timestamp'}.issubset(ref.columns)
assert {'event_a_id','event_b_id','issue_key','relation'}.issubset(rels.columns)
assert ref.event_id.is_unique
ref['issue_key']=ref.issue_key.str.upper().str.strip()
ref['ts']=pd.to_datetime(ref.timestamp,format='mixed',utc=True,errors='coerce')
assert ref.ts.notna().all(),'Unparseable reference timestamps: inspect source; do not fill.'
assert set(ref.project_key)==set(PROJECTS)
assert set(rels.event_a_id).issubset(ref.event_id) and set(rels.event_b_id).issubset(ref.event_id)
print('Original reference:',len(ref),'events;',len(rels),'relations;',ref.issue_key.nunique(),'issues')
display(ref.event_type.value_counts().rename_axis('event_type').reset_index(name='count'))

Original reference: 1208 events; 1148 relations; 60 issues


,event_type,count
0,STATUS_CHANGED,504
1,FIX_VERSION_CHANGED,323
2,ASSIGNEE_CHANGED,160
3,REOPENED,76
4,ISSUE_CREATED,60
5,ISSUE_RESOLVED,45
6,COMPONENT_CHANGED,26
7,PRIORITY_CHANGED,13
8,DUE_DATE_CHANGED,1


## 2. Reproduce the instance-window eligibility audit from original Jira records

The curated Jira format is **flattened**, using `created_at`, `status_history[].changed_at`, and `full_changelog[].created` (not `fields` or `changelog.histories`). We use the same broad timestamp inventory as the prior corrected check. This validates alignment with the instance-window rule; it is not a claim that every timestamp is an extractable event.

In [3]:
def read_jsonl_file(path):
    with open(path,encoding='utf-8-sig') as f:
        for line in f:
            if line.strip():yield json.loads(line)

def parse_one(v):
    if v is None or v=='':return pd.NaT
    return pd.to_datetime(v,format='mixed',utc=True,errors='coerce')

def activity_times(t):
    vals=[t.get(k) for k in ['created_at','updated_at','resolved_at','due_at','blocked_at','unblocked_at']]
    vals += [x.get('changed_at') for x in (t.get('status_history') or []) if isinstance(x,dict)]
    vals += [x.get('created') for x in (t.get('full_changelog') or []) if isinstance(x,dict)]
    return [v for v in (parse_one(x) for x in vals) if pd.notna(v)]

issue_ids=set(ref.issue_key)
windows={}; audit=[]
for project in PROJECTS:
    folders=[p for p in PRD.iterdir() if p.is_dir() and p.name.endswith('_'+project)]
    assert len(folders)==1,f'Ambiguous project folder: {project}'
    manifest=folders[0]/f'{project}-I01'/'snapshot_manifest.json'
    assert manifest.exists(),f'Missing I01 manifest: {manifest}'
    meta=json.loads(manifest.read_text(encoding='utf-8'))
    start,end=parse_one(meta['instance_start']),parse_one(meta['instance_end'])
    assert pd.notna(start) and pd.notna(end) and start<=end
    windows[project]=(start,end)
    jira_files=list(JIRA.glob(f'*_{project}/jira_tickets.jsonl'))
    assert len(jira_files)==1,f'Expected one curated Jira input for {project}: {jira_files}'
    tickets={str(t.get('issue_key','')).strip().upper():t for t in read_jsonl_file(jira_files[0])}
    for key in sorted(k for k in issue_ids if k.startswith(project+'-')):
        assert key in tickets,f'Reference issue absent from curated input: {key}'
        times=activity_times(tickets[key]); active=any(start<=t<=end for t in times)
        audit.append({'issue_key':key,'project':project,'in_scope':active,'activity_count':len(times),
                      'window_start':start.isoformat(),'window_end':end.isoformat()})
window_audit=pd.DataFrame(audit)
assert len(window_audit)==60 and window_audit.issue_key.is_unique
print('Ticket eligibility:');display(pd.crosstab(window_audit.project,window_audit.in_scope,margins=True))
print('In-scope:',int(window_audit.in_scope.sum()),'Out-of-scope:',int((~window_audit.in_scope).sum()))
window_audit.to_csv(OUT/'ticket_instance_eligibility_60.csv',index=False)
# Important: fail visibly if scope no longer matches the audited snapshot.
if window_audit.in_scope.sum()!=35:
    raise AssertionError('Expected 35 eligible tickets based on prior audit. Investigate differences before scoring.')

Ticket eligibility:


in_scope,False,True,All
project,,,
HBASE,6,4,10
JBIDE,3,7,10
MC,4,6,10
MNG,3,7,10
SAK,4,6,10
SERVER,5,5,10
All,25,35,60


In-scope: 35 Out-of-scope: 25


## 3. Restrict reference events to eligible tickets **and** instance dates

An eligible ticket can have older/later events. Count those separately; do not score out-of-window events as omissions.

In [4]:
eligible=set(window_audit.loc[window_audit.in_scope,'issue_key'])
ref['ticket_eligible']=ref.issue_key.isin(eligible)
ref['event_in_window']=ref.apply(lambda x: windows[x.project_key][0]<=x.ts<=windows[x.project_key][1],axis=1)
ref['in_scope']=ref.ticket_eligible & ref.event_in_window
r=ref[ref.in_scope].copy()
print('All reference events:',len(ref),'on eligible tickets:',int(ref.ticket_eligible.sum()),'in-window events:',len(r))
print('Out-of-scope events:',int((~ref.in_scope).sum()))
display(pd.crosstab(ref.event_type,ref.in_scope).rename(columns={False:'excluded',True:'in_scope'}))
assert len(r)>0
r.to_csv(OUT/'reference_events_in_scope.csv',index=False)
ref[['event_id','issue_key','event_type','timestamp','ticket_eligible','event_in_window','in_scope']].to_csv(OUT/'reference_scope_audit.csv',index=False)

All reference events: 1208 on eligible tickets: 840 in-window events: 134
Out-of-scope events: 1074


in_scope,excluded,in_scope
event_type,,
ASSIGNEE_CHANGED,154,6
COMPONENT_CHANGED,26,0
DUE_DATE_CHANGED,1,0
FIX_VERSION_CHANGED,306,17
ISSUE_CREATED,53,7
ISSUE_RESOLVED,42,3
PRIORITY_CHANGED,13,0
REOPENED,70,6
STATUS_CHANGED,409,95


## 4. Read actual Event-node schema from Temporal KG v2

The exported graph uses `source_id` for the Jira issue, `event_time` for the event timestamp and `payload_json.event_type` for the event vocabulary.

In [5]:
def load_jsonl(path_or_zip,filename):
    if path_or_zip.is_dir():
        files=list(path_or_zip.rglob(filename))
        if len(files)!=1:raise FileNotFoundError(f'Expected one {filename}, found {len(files)}')
        with open(files[0],encoding='utf-8-sig') as f:
            for line in f:
                if line.strip():yield json.loads(line)
    else:
        with zipfile.ZipFile(path_or_zip) as z:
            files=[x for x in z.namelist() if x.endswith('/'+filename) or x==filename]
            if len(files)!=1:raise FileNotFoundError(f'Expected one {filename} in zip, found {len(files)}')
            with z.open(files[0]) as f:
                for line in f:
                    if line.strip():yield json.loads(line)

rows=[];node_counts=collections.Counter();event_uids=set()
for x in load_jsonl(GRAPH,'nodes.jsonl'):
    node_counts[x.get('label')]+=1
    if x.get('label')!='Event':continue
    event_uids.add(x.get('uid'))
    try:p=json.loads(x.get('payload_json') or '{}')
    except (TypeError,ValueError):p={}
    if not isinstance(p,dict):p={}
    attrs=p.get('attributes') or {}
    if not isinstance(attrs,dict):attrs={}
    rows.append({'uid':x.get('uid'),'issue_key':str(x.get('source_id') or '').upper().strip(),
                 'raw_type':str(p.get('event_type') or '').upper().strip(),
                 'event_time':x.get('event_time') or p.get('event_time') or '',
                 'observed_at':x.get('observed_at') or '',
                 'field':str(attrs.get('jira_field') or ''),
                 'instance_id':x.get('instance_id') or '',
                 'temporal_status':x.get('temporal_status') or ''})
g=pd.DataFrame(rows)
g['ts']=pd.to_datetime(g.event_time,format='mixed',utc=True,errors='coerce')
g['obs_ts']=pd.to_datetime(g.observed_at,format='mixed',utc=True,errors='coerce')
g['project']=g.issue_key.str.split('-').str[0]
g['ticket_eligible']=g.issue_key.isin(eligible)
g['event_in_window']=g.apply(lambda x: (x.project in windows and pd.notna(x.ts) and windows[x.project][0]<=x.ts<=windows[x.project][1]),axis=1)
g['in_scope']=g.ticket_eligible & g.event_in_window
pg=g[g.in_scope].copy()
print('Graph node kinds:',dict(node_counts))
print('Graph events overall:',len(g),'on original 60 tickets:',int(g.issue_key.isin(issue_ids).sum()),'in scope:',len(pg))
print('Eligible tickets with Event nodes:',pg.issue_key.nunique(),'of',len(eligible))
display(pg.raw_type.value_counts().head(25).rename_axis('graph_event_type').reset_index(name='count'))
g[g.issue_key.isin(issue_ids)].to_csv(OUT/'graph_events_60_scope_audit.csv',index=False)

Graph node kinds: {'CanonicalEntity': 40325, 'SourceRecord': 5827, 'Evidence': 5827, 'Dependency': 7709, 'Fact': 63305, 'Event': 82577, 'Decision': 414, 'ActionItem': 207, 'Blocker': 443}
Graph events overall: 82577 on original 60 tickets: 1408 in scope: 1408
Eligible tickets with Event nodes: 35 of 35


,graph_event_type,count
0,LINK_CHANGE,645
1,STATUS_CHANGE,222
2,VERSION_CHANGE,147
3,COMMENT_CHANGE,97
4,RANK_CHANGE,79
5,REMOTE_LINK_CHANGE,76
6,ATTACHMENT_CHANGE,65
7,DESCRIPTION_CHANGE,17
8,RESOLUTION_CHANGE,16
9,LABEL_CHANGE,9


## 5. Predeclare justified event mappings

The supported vocabulary is fixed **before** examining evaluation matches. We do not infer creation, reopening or resolution events from other categories. `VERSION_CHANGE` maps to `FIX_VERSION_CHANGED` only when its Jira field explicitly identifies fix versions. All unsupported reference types and unmapped graph types remain visible in the coverage audit.


In [6]:
DIRECT={'STATUS_CHANGE':'STATUS_CHANGED','ASSIGNEE_CHANGE':'ASSIGNEE_CHANGED',
        'COMPONENT_CHANGE':'COMPONENT_CHANGED','PRIORITY_CHANGE':'PRIORITY_CHANGED',
        'DUE_DATE_CHANGE':'DUE_DATE_CHANGED'}
def align(row):
    t=row.raw_type;field=str(row.field).lower().strip()
    if t in DIRECT:return DIRECT[t]
    if t=='VERSION_CHANGE' and field in {'fix version','fix versions','fixversion','fixversions'}:
        return 'FIX_VERSION_CHANGED'
    return ''
pg['aligned_type']=pg.apply(align,axis=1)
SUPPORTED=set(DIRECT.values())|{'FIX_VERSION_CHANGED'} # declared mapping, not types discovered in graph
r['etype']=r.event_type.str.upper().str.strip()
print('In-scope reference by type:');display(r.etype.value_counts().to_frame('reference'))
print('Graph mapped predictions:',int(pg.aligned_type.ne('').sum()),'unmapped:',int(pg.aligned_type.eq('').sum()))
display(pg.loc[pg.aligned_type.eq(''),'raw_type'].value_counts().head(20).to_frame('unmapped'))
print('Graph VERSION_CHANGE fields:');display(pg.loc[pg.raw_type.eq('VERSION_CHANGE'),'field'].value_counts().to_frame('count'))
print('Reference types with no justified mapping:',sorted(set(r.etype)-SUPPORTED))
pg.to_csv(OUT/'graph_events_in_scope_aligned.csv',index=False)
# Document fixed vocabulary and uncovered categories for reproducibility.
vocabulary_audit = pd.DataFrame([{'graph_type':k,'reference_type':v,'rule':'direct label equivalence'} for k,v in DIRECT.items()] + [{'graph_type':'VERSION_CHANGE (jira_field=Fix Version)','reference_type':'FIX_VERSION_CHANGED','rule':'field-qualified equivalence'}])
vocabulary_audit.to_csv(OUT/'V4_vocabulary_mapping.csv',index=False)
ref_type_coverage = r.groupby('etype').size().rename('reference_events').reset_index()
ref_type_coverage['supported'] = ref_type_coverage.etype.isin(SUPPORTED)
ref_type_coverage.to_csv(OUT/'V4_reference_type_coverage.csv',index=False)


In-scope reference by type:


,reference
etype,
STATUS_CHANGED,95
FIX_VERSION_CHANGED,17
ISSUE_CREATED,7
REOPENED,6
ASSIGNEE_CHANGED,6
ISSUE_RESOLVED,3


Graph mapped predictions: 246 unmapped: 1162


,unmapped
raw_type,
LINK_CHANGE,645
VERSION_CHANGE,130
COMMENT_CHANGE,97
RANK_CHANGE,79
REMOTE_LINK_CHANGE,76
ATTACHMENT_CHANGE,65
DESCRIPTION_CHANGE,17
RESOLUTION_CHANGE,16
LABEL_CHANGE,9


Graph VERSION_CHANGE fields:


,count
field,
Version,130
Fix Version,17


Reference types with no justified mapping: ['ISSUE_CREATED', 'ISSUE_RESOLVED', 'REOPENED']


## 6. Strict event evaluation on comparable categories

Match one-to-one by **issue + event type + exact UTC timestamp** using multisets, so duplicate events cannot receive extra credit. Denominator includes all in-scope reference events of supported types, even if the graph produced none for that type. Precision is conditional on mapped graph categories, not precision across all graph events.

In [7]:
def keys(frame,typecol):
    return collections.Counter((x.issue_key,str(getattr(x,typecol)),x.ts.isoformat())
      for x in frame.itertuples(index=False) if pd.notna(x.ts))
def prf(tp,fp,fn):
    precision=tp/(tp+fp) if tp+fp else np.nan
    recall=tp/(tp+fn) if tp+fn else np.nan
    f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan
    return precision,recall,f1
rsub=r[r.etype.isin(SUPPORTED)].copy()
psub=pg[pg.aligned_type.ne('') & pg.ts.notna()].copy()
R=keys(rsub,'etype');P=keys(psub,'aligned_type')
tp=sum((R&P).values());fp=sum((P-R).values());fn=sum((R-P).values())
p,rec,f1=prf(tp,fp,fn)
metrics=pd.DataFrame([{'evaluation':'in-scope, directly comparable event types','eligible_tickets':len(eligible),
 'reference_events_in_scope_all':len(r),'reference_events_supported':len(rsub),
 'graph_events_in_scope_all':len(pg),'graph_events_mapped':len(psub),
 'TP':tp,'FP':fp,'FN':fn,'precision':p,'recall':rec,'F1':f1,
 'supported_reference_fraction':len(rsub)/len(r),
 'strict_match_fraction_of_all_in_scope_reference':tp/len(r)}])
display(metrics.round(4));metrics.to_csv(OUT/'V3_event_metrics.csv',index=False)
rows=[]
for typ in sorted(SUPPORTED):
    a=collections.Counter({k:v for k,v in R.items() if k[1]==typ})
    b=collections.Counter({k:v for k,v in P.items() if k[1]==typ})
    t=sum((a&b).values());f=sum((b-a).values());n=sum((a-b).values())
    pp,rr,ff=prf(t,f,n)
    rows.append({'event_type':typ,'reference':sum(a.values()),'predicted':sum(b.values()),'TP':t,'FP':f,'FN':n,'precision':pp,'recall':rr,'F1':ff})
per_type=pd.DataFrame(rows)
display(per_type.round(4));per_type.to_csv(OUT/'V3_event_metrics_by_type.csv',index=False)
missing=[{'issue_key':k[0],'type':k[1],'timestamp':k[2],'count':v} for k,v in (R-P).items()]
extra=[{'issue_key':k[0],'type':k[1],'timestamp':k[2],'count':v} for k,v in (P-R).items()]
pd.DataFrame(missing,columns=['issue_key','type','timestamp','count']).to_csv(OUT/'V3_unmatched_reference.csv',index=False)
pd.DataFrame(extra,columns=['issue_key','type','timestamp','count']).to_csv(OUT/'V3_unmatched_graph.csv',index=False)
print('Exact matching errors saved for audit.')

,evaluation,eligible_tickets,reference_events_in_scope_all,reference_events_supported,graph_events_in_scope_all,graph_events_mapped,TP,FP,FN,precision,recall,F1,supported_reference_fraction,strict_match_fraction_of_all_in_scope_reference
0,"in-scope, directly comparable event types",35,134,118,1408,246,61,185,57,0.248,0.5169,0.3352,0.8806,0.4552


,event_type,reference,predicted,TP,FP,FN,precision,recall,F1
0,ASSIGNEE_CHANGED,6,7,3,4,3,0.4286,0.5000,0.4615
1,COMPONENT_CHANGED,0,0,0,0,0,NaN,NaN,NaN
2,DUE_DATE_CHANGED,0,0,0,0,0,NaN,NaN,NaN
3,FIX_VERSION_CHANGED,17,17,5,12,12,0.2941,0.2941,0.2941
4,PRIORITY_CHANGED,0,0,0,0,0,NaN,NaN,NaN
5,STATUS_CHANGED,95,222,53,169,42,0.2387,0.5579,0.3344


Exact matching errors saved for audit.


## 7. Project breakdown and all-60 traceability

Project scores can be undefined where no supported positives or predictions exist. Ticket coverage is **not** event recall.

In [8]:
proj_rows=[]
for project in PROJECTS:
    a=collections.Counter({k:v for k,v in R.items() if k[0].startswith(project+'-')})
    b=collections.Counter({k:v for k,v in P.items() if k[0].startswith(project+'-')})
    t=sum((a&b).values());f=sum((b-a).values());n=sum((a-b).values())
    pp,rr,ff=prf(t,f,n)
    proj_rows.append({'project':project,'eligible_tickets':int(window_audit[(window_audit.project==project)&window_audit.in_scope].shape[0]),
      'reference_supported_events':sum(a.values()),'graph_mapped_events':sum(b.values()),'TP':t,'FP':f,'FN':n,'precision':pp,'recall':rr,'F1':ff})
by_project=pd.DataFrame(proj_rows)
display(by_project.round(4));by_project.to_csv(OUT/'V3_event_metrics_by_project.csv',index=False)
coverage=window_audit.copy()
coverage['graph_event_nodes']=coverage.issue_key.map(g[g.issue_key.isin(issue_ids)].groupby('issue_key').size()).fillna(0).astype(int)
coverage['reference_event_rows_all_dates']=coverage.issue_key.map(ref.groupby('issue_key').size()).fillna(0).astype(int)
coverage['reference_event_rows_in_window']=coverage.issue_key.map(r.groupby('issue_key').size()).fillna(0).astype(int)
coverage['graph_event_rows_in_window']=coverage.issue_key.map(pg.groupby('issue_key').size()).fillna(0).astype(int)
display(coverage.groupby('in_scope')[['graph_event_nodes','reference_event_rows_all_dates','reference_event_rows_in_window','graph_event_rows_in_window']].sum())
coverage.to_csv(OUT/'V3_60_ticket_coverage_audit.csv',index=False)

,project,eligible_tickets,reference_supported_events,graph_mapped_events,TP,FP,FN,precision,recall,F1
0,HBASE,4,38,43,0,43,38,0.0000,0.0,0.0000
1,JBIDE,7,0,0,0,0,0,NaN,NaN,NaN
2,MC,6,0,121,0,121,0,0.0000,NaN,0.0000
3,MNG,7,19,19,0,19,19,0.0000,0.0,0.0000
4,SAK,6,2,2,2,0,0,1.0000,1.0,1.0000
5,SERVER,5,59,61,59,2,0,0.9672,1.0,0.9833


,graph_event_nodes,reference_event_rows_all_dates,reference_event_rows_in_window,graph_event_rows_in_window
in_scope,,,,
False,0,368,0,0
True,1408,840,134,1408


## 8. Consecutive-event ordering diagnostic

The reference has `BEFORE` and `SIMULTANEOUS` consecutive relations. The graph export has no explicit `NEXT_EVENT`/`BEFORE` edges. This diagnostic checks only uniquely matched reference events and **cannot be interpreted as graph relation extraction F1**. Because strict event matching already requires equal timestamps, ordering agreement is partly tautological; report its coverage, not as independent validation of chronology.

In [9]:
rsub=rsub.copy();psub=psub.copy()
rsub['match_key']=list(zip(rsub.issue_key,rsub.etype,rsub.ts.map(lambda t:t.isoformat())))
psub['match_key']=list(zip(psub.issue_key,psub.aligned_type,psub.ts.map(lambda t:t.isoformat())))
rc=rsub.match_key.value_counts();pc=psub.match_key.value_counts()
unique=set(rc[rc==1].index)&set(pc[pc==1].index)
ref_to_key={x.event_id:x.match_key for x in rsub.itertuples(index=False) if x.match_key in unique}
rel_scope=rels[rels.event_a_id.isin(set(r.event_id)) & rels.event_b_id.isin(set(r.event_id))].copy()
matched=rel_scope[rel_scope.event_a_id.isin(ref_to_key)&rel_scope.event_b_id.isin(ref_to_key)].copy()
if len(matched):
    matched['graph_a_time']=matched.event_a_id.map(lambda x:pd.Timestamp(ref_to_key[x][2]))
    matched['graph_b_time']=matched.event_b_id.map(lambda x:pd.Timestamp(ref_to_key[x][2]))
    matched['derived_relation']=np.where(matched.graph_a_time<matched.graph_b_time,'BEFORE',np.where(matched.graph_a_time==matched.graph_b_time,'SIMULTANEOUS','AFTER'))
    matched['agrees']=matched.derived_relation.eq(matched.relation)
    agreement=float(matched.agrees.mean())
else:agreement=np.nan
order=pd.DataFrame([{'original_reference_relations':len(rels),'in_scope_reference_relations':len(rel_scope),
 'relations_with_both_events_uniquely_matched':len(matched),'conditional_order_agreement':agreement,
 'fraction_of_in_scope_relations_testable':len(matched)/len(rel_scope) if len(rel_scope) else np.nan,
 'note':'Derived from exact matched timestamps; not an independent relation extraction score'}])
display(order);order.to_csv(OUT/'V3_ordering_diagnostic.csv',index=False)
matched.to_csv(OUT/'V3_ordering_matched_pairs.csv',index=False)

,original_reference_relations,in_scope_reference_relations,relations_with_both_events_uniquely_matched,conditional_order_agreement,fraction_of_in_scope_relations_testable,note
0,1148,121,53,1.0,0.438017,Derived from exact matched timestamps; not an ...


## 9. Point-in-time availability and structural checks

`observed_at` is not the same as event occurrence time. Audit coverage and suspicious ordering, but does not establish leakage-free downstream risk prediction without testing feature queries at historical cutoffs.

In [10]:
audit=pd.DataFrame([{'graph_event_nodes_in_scope':len(pg),
 'event_time_parseable':int(pg.ts.notna().sum()),
 'observed_at_parseable':int(pg.obs_ts.notna().sum()),
 'observed_at_missing':int(pg.obs_ts.isna().sum()),
 'observed_before_event_time':int(((pg.obs_ts<pg.ts)&pg.obs_ts.notna()&pg.ts.notna()).sum()),
 'temporal_status_unknown':int(pg.temporal_status.str.upper().eq('UNKNOWN').sum()),
 'note':'Metadata audit only; historical feature queries not tested'}])
display(audit);audit.to_csv(OUT/'V3_availability_audit.csv',index=False)
link_counts=collections.Counter();has_event=0;dangling=0
for edge in load_jsonl(GRAPH,'links.jsonl'):
    kind=edge.get('type');link_counts[kind]+=1
    if kind=='HAS_EVENT':
        has_event+=1
        if edge.get('to_uid') not in event_uids:dangling+=1
print('HAS_EVENT edges:',has_event,'dangling targets:',dangling)
assert dangling==0,'Graph has dangling HAS_EVENT targets; investigate.'
pd.DataFrame([{'link_type':k,'count':v} for k,v in link_counts.items()]).to_csv(OUT/'V3_graph_link_inventory.csv',index=False)
# Availability-at-cutoff audit: metadata completeness only, not feature-query testing.
print('Unknown observed_at (in-scope):',int(pg.obs_ts.isna().sum()))
print('Event timestamps invalid on eligible tickets (excluded from in-window scoring):',int((g.ticket_eligible & g.ts.isna()).sum()))


,graph_event_nodes_in_scope,event_time_parseable,observed_at_parseable,observed_at_missing,observed_before_event_time,temporal_status_unknown,note
0,1408,1408,1408,0,0,0,Metadata audit only; historical feature querie...


HAS_EVENT edges: 82577 dangling targets: 0
Unknown observed_at (in-scope): 0
Event timestamps invalid on eligible tickets (excluded from in-window scoring): 0


## 10. Final reproducibility summary and interpretation

In [11]:
summary={
 'benchmark':'Temporal KG V4 — final methodology candidate',
 'reference_issues_original':60,'eligible_issues':len(eligible),'out_of_scope_issues':60-len(eligible),
 'reference_events_original':len(ref),'reference_events_in_scope':len(r),
 'reference_relations_original':len(rels),'reference_relations_in_scope':len(rel_scope),
 'graph_event_nodes_in_scope':len(pg),'graph_eligible_issues_with_events':pg.issue_key.nunique(),
 'mapped_reference_events':len(rsub),'mapped_graph_events':len(psub),
 'TP':tp,'FP':fp,'FN':fn,'precision':None if pd.isna(p) else float(p),
 'recall':None if pd.isna(rec) else float(rec),'F1':None if pd.isna(f1) else float(f1),
 'ordering_matched_pairs':len(matched),'ordering_fraction_testable':len(matched)/len(rel_scope) if len(rel_scope) else None,'ordering_conditional_agreement':None if pd.isna(agreement) else agreement,
 'fixed_supported_types':sorted(SUPPORTED),
 'scope_policy':'Ticket eligibility: activity in I01 window; event eligibility: event timestamp in same window',
 'limitations':[
  'Reference is Jira-derived and spot-checked, not entirely human-annotated.',
  'Event F1 conditional on documented event-vocabulary equivalences.',
  'Graph events from unmapped categories are reported separately, not counted as false positives.',
  'Derived ordering agreement on exact timestamp matches is not independent evidence of relation extraction.',
  'Availability metadata audit does not establish downstream leakage freedom.',
  'Selection of benchmark tickets originally preceded this scope correction; generalization beyond this benchmark is not established.',
  'Out-of-scope 25 tickets retained in the 60-ticket traceability audit; not scored in-scope.'
 ]}
(OUT/'V4_methodology_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))
print('All V4 outputs saved to:',OUT)

{
  "benchmark": "Temporal KG V4 \u2014 final methodology candidate",
  "reference_issues_original": 60,
  "eligible_issues": 35,
  "out_of_scope_issues": 25,
  "reference_events_original": 1208,
  "reference_events_in_scope": 134,
  "reference_relations_original": 1148,
  "reference_relations_in_scope": 121,
  "graph_event_nodes_in_scope": 1408,
  "graph_eligible_issues_with_events": 35,
  "mapped_reference_events": 118,
  "mapped_graph_events": 246,
  "TP": 61,
  "FP": 185,
  "FN": 57,
  "precision": 0.24796747967479674,
  "recall": 0.5169491525423728,
  "F1": 0.33516483516483514,
  "ordering_matched_pairs": 53,
  "ordering_fraction_testable": 0.4380165289256198,
  "ordering_conditional_agreement": 1.0,
  "fixed_supported_types": [
    "ASSIGNEE_CHANGED",
    "COMPONENT_CHANGED",
    "DUE_DATE_CHANGED",
    "FIX_VERSION_CHANGED",
    "PRIORITY_CHANGED",
    "STATUS_CHANGED"
  ],
  "scope_policy": "Ticket eligibility: activity in I01 window; event eligibility: event timestamp in same 